# 模型初始測試

In [1]:
API_KEY = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
model_name = 'gemini-2.5-flash'

In [2]:
from google import genai

client = genai.Client(api_key=API_KEY)

prompt = "機器學習的定義"
response = client.models.generate_content(
    model=model_name,
    contents=prompt,
)

print(response.text)

機器學習 (Machine Learning, ML) 是**人工智慧 (Artificial Intelligence, AI) 的一個子領域**，其核心思想是讓電腦系統能夠**從數據中學習並識別模式**，而**無需經過明確的程式設計**來執行特定任務。

簡單來說，它的定義可以概括為：

**機器學習是讓電腦系統透過分析大量的數據樣本（稱為『經驗』），自動地構建一個『模型』，這個模型能夠捕捉數據中潛在的規律、關係和結構，然後利用這些學習到的知識來進行預測、分類、決策或發現新的洞察。隨著經驗（更多數據）的增加，機器學習系統的性能通常會不斷提高。**

**核心要點：**

1.  **學習而非編程：** 與傳統的程式設計不同，傳統程式需要開發者為每個可能的情況編寫詳細的規則和指令。機器學習則是透過提供數據和一個學習算法，讓電腦自己去發現這些規則。
2.  **從數據中學習：** 數據是機器學習的燃料。系統從這些數據中「學習」模式、關聯性或結構。
3.  **構建模型：** 學習的結果是一個數學模型。這個模型代表了從數據中提取的知識。
4.  **預測、分類和決策：** 學習到的模型會被用來對新的、未見過的數據做出預測（例如預測房價）、進行分類（例如識別圖片中的物體）或輔助決策（例如推薦商品）。
5.  **性能提升：** 學習的過程通常伴隨著性能的評估，系統會嘗試優化其模型以提高在特定任務上的表現。

**主要範疇包括：**

*   **監督式學習 (Supervised Learning)：** 從帶有標籤的數據中學習，以預測輸出（如分類或迴歸）。
*   **無監督式學習 (Unsupervised Learning)：** 從無標籤數據中發現隱藏的模式或結構（如聚類或降維）。
*   **強化學習 (Reinforcement Learning)：** 透過與環境互動，從獎勵和懲罰中學習最佳行為策略。

機器學習已廣泛應用於多個領域，例如圖像識別、自然語言處理、推薦系統、垃圾郵件過濾、醫療診斷、自動駕駛等。


# 開始測試

In [ ]:
# ! pip install geopy

In [3]:
from geopy.geocoders import Nominatim

def get_coordinates(city_name):
    geolocator = Nominatim(user_agent="clement@example.com")
    location = geolocator.geocode(city_name)
    if location:
        return (location.latitude, location.longitude)
    else:
        return None


# get_coordinates("Taipei")
get_coordinates("台北")

(25.0375198, 121.5636796)

In [4]:
import requests

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

get_weather(25.047554, 121.5170503) #台北市溫度

34.8

# 格式定義參考
- https://ai.google.dev/api/caching#Schema
- https://spec.openapis.org/oas/v3.0.3#schema

In [5]:
get_coordinates_declaration = {
        "name": "get_coordinates",
        "description": "取得城市的GPS座標",
        "parameters": {
            "type": "object",
            "properties": {
                "city_name": { "type": "string", "description": "城市名稱" }
            },
        },
    }

get_weather_declaration = {
        "name": "get_weather",
        "description": "取得溫度值",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": { "type": "number", "description": "GPS經度"},
                "longitude": { "type": "number", "description": "GPS緯度" }
            }
        },
    }

In [6]:
from google import genai
from google.genai import types
from google.genai.types import Content, Part  # 引入 Content 類型

# Configure the client
client = genai.Client(api_key=API_KEY)

# Generation Config with Function Declaration
tools = types.Tool(function_declarations=[get_coordinates_declaration, get_weather_declaration])
config = types.GenerateContentConfig(tools=[tools])

system_prompt = '''
你是一個個人助理。你的任務是簡潔的回答使用者的問題。
當你需要使用工具時，可以直接選擇使用，不需要詢問使用者。
'''
contents = [
    Content(role="user", parts=[Part(text=system_prompt)]),
    Content(role="user", parts=[Part(text="紐約今天適合出門玩嗎？")]),
]

# # 測試LLM能不能正確的選擇工具
response = client.models.generate_content(
    model=model_name, config=config, contents=contents
)
response.candidates[0].content

Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'city_name': '紐約'
        },
        name='get_coordinates'
      ),
      thought_signature=b'\n\xb8\x03\x01\xd1\xed\x8ao\xad\x138W\xbf\x91\xc8\xb4\xab%Gs$>\x85\x10\xc2\x8a.\xc8y\xb5\xb0\xf4\xc5r\xe4&\xf3\\\xd7W{\x86>D\xa5\xea}\x12J\xe2\\R\x91\xd9\xa6\xefK\x0c>\xb8\xd1\x05\xfa\xe8e"\x0c\x19x\x91\x81Z<\xb4\xa7Ud\xe2\xb3\x9eU\xa2\x12/\xd0\x08\xff\x81\xa7\xb6A0g\xe0\xcf=\xad...'
    ),
  ],
  role='model'
)

In [7]:
# 檢查function_call內容
response.candidates[0].content.parts[0].function_call

FunctionCall(
  args={
    'city_name': '紐約'
  },
  name='get_coordinates'
)

In [8]:
# 代為執行程式
tool_call = response.candidates[0].content.parts[0].function_call
result = get_coordinates(**tool_call.args)
result

(40.7127281, -74.0060152)

In [9]:
# 加入模型剛剛的訊息到訊息列
contents.append(response.candidates[0].content)  # append model's function call message

# 將執行結果加入訊息列
function_response_part = types.Part.from_function_response(
    name=tool_call.name,
    response={"result": result},
)
contents.append(types.Content(role="user", parts=[function_response_part])) # Append the function response

# 再次呼叫LLM
response = client.models.generate_content(
    model=model_name, config=config, contents=contents
)
response.candidates[0].content

Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'latitude': 40.7127281,
          'longitude': -74.0060152
        },
        name='get_weather'
      ),
      thought_signature=b'\n\xf1\x03\x01\xd1\xed\x8ao\xf8\x98\xebi\xd1\x8f*\xa2=\xef_\xb2\x85\x15_^k\x07\x9dyn]\xa8,\xb7\r"\x19\x18\xeb\xc3\xc1\x18\x01\x00%\x1a}]\xa5\x04\x10dP\x98\x07\xb9\x83{(\x04\xa7\xcfzg\xc2\xc9Z\xb8\x80\xf4\x01\xc9\x9dW\xba\x0f.\xa3\x9f\x07\x17r\xd1+zJ\xa4\xadF\x89|\'\xfa\xbe\xb6\x1c\x9b\x19...'
    ),
  ],
  role='model'
)

In [10]:
# 代為執行程式
tool_call = response.candidates[0].content.parts[0].function_call
result = get_weather(**tool_call.args)
result

17.5

In [11]:
contents

[Content(
   parts=[
     Part(
       text="""
 你是一個個人助理。你的任務是簡潔的回答使用者的問題。
 當你需要使用工具時，可以直接選擇使用，不需要詢問使用者。
 """
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       text='紐約今天適合出門玩嗎？'
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'city_name': '紐約'
         },
         name='get_coordinates'
       ),
       thought_signature=b'\n\xb8\x03\x01\xd1\xed\x8ao\xad\x138W\xbf\x91\xc8\xb4\xab%Gs$>\x85\x10\xc2\x8a.\xc8y\xb5\xb0\xf4\xc5r\xe4&\xf3\\\xd7W{\x86>D\xa5\xea}\x12J\xe2\\R\x91\xd9\xa6\xefK\x0c>\xb8\xd1\x05\xfa\xe8e"\x0c\x19x\x91\x81Z<\xb4\xa7Ud\xe2\xb3\x9eU\xa2\x12/\xd0\x08\xff\x81\xa7\xb6A0g\xe0\xcf=\xad...'
     ),
   ],
   role='model'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         name='get_coordinates',
         response={
           'result': (
             40.7127281,
             -74.0060152,
           )
         }
       )
     ),
   ],
   r

In [12]:
contents.append(response.candidates[0].content)  # append model's function call message

function_response_part = types.Part.from_function_response(
    name=tool_call.name,
    response={"result": result},
)
contents.append(types.Content(role="user", parts=[function_response_part])) # Append the function response

response = client.models.generate_content(
    model=model_name, config=config, contents=contents
)
response.candidates[0].content

Content(
  parts=[
    Part(
      text='紐約今天氣溫為17.5攝氏度，非常適合出門活動。',
      thought_signature=b'\n\xb0\x03\x01\xd1\xed\x8ao\x1a\xc4\xa0<N\xd9\xc9\x99d\x89f\xc4\xf3t\xef@\xbd\x81\x1f\x7f\xdf\xa6\xdb\xdaa\xbe\xd1&\xddV\xef\x01\xcbg`u\xf8\x8e\x00\x8b\xa4L\x17.n\x9f\x8dS#\xac5S\x99\'\x9bvG\x82;\xa1\xa1\xfd"$\xbe\xae\xccX\xe0\xf8\xb3>\xe3\x92\xe1\xcb^\x05\x9b\x8635\x04\x0e\xcb\x16\xa5\xdf\xa9...'
    ),
  ],
  role='model'
)

# 將流程串起來

In [13]:
from geopy.geocoders import Nominatim

def get_coordinates(city_name):
    geolocator = Nominatim(user_agent="clement@example.com")
    location = geolocator.geocode(city_name)
    if location:
        return (location.latitude, location.longitude)
    else:
        return None

import requests
def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']


get_coordinates_declaration = {
        "name": "get_coordinates",
        "description": "取得城市的GPS座標",
        "parameters": {
            "type": "object",
            "properties": {
                "city_name": { "type": "string", "description": "城市名稱" }
            },
        },
    }

get_weather_declaration = {
        "name": "get_weather",
        "description": "取得溫度值",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": { "type": "number", "description": "GPS經度"},
                "longitude": { "type": "number", "description": "GPS緯度" }
            }
        },
    }

from google import genai
from google.genai import types
from google.genai.types import Content, Part

tools = types.Tool(function_declarations=[get_coordinates_declaration, get_weather_declaration])
config = types.GenerateContentConfig(tools=[tools])

In [14]:
class weather_bot:
    def __init__(self, API_KEY, model_name, config):
        self.client = genai.Client(api_key=API_KEY)
        self.model_name = model_name
        self.config = config
        system_prompt = '''
            你是一個個人助理。你的任務是簡潔的回答使用者的問題。
            當你需要使用工具時，可以直接選擇使用，不需要詢問使用者。
            '''
        self.contents = contents = [
            Content(role="user", parts=[Part(text=system_prompt)]),
        ]

    def call_function(self, tool_call):
        # print('call: ' + tool_call.name)
        try:
            return globals()[tool_call.name](**tool_call.args)
        except:
            return f"Call {tool_call.name} 失敗"
    
    # def call_function(self, tool_call):
    #     print('call: ' + tool_call.name)
    #     if tool_call.name == 'get_weather':
    #         return get_weather(**tool_call.args)
    #     elif tool_call.name == 'get_coordinates':
    #         return get_coordinates(**tool_call.args)
    #     else:
    #         return None
        
    def chat(self, text):
        history = ''
        self.contents.append(Content(role="user", parts=[Part(text=text)]))
        
        while True:
            response = client.models.generate_content(model=self.model_name, config=self.config, contents=self.contents)
            self.contents.append(response.candidates[0].content)

            result_parts = []
            is_tools_call = False
            for part in response.candidates[0].content.parts:
                if part.function_call:
                    is_tools_call = True
                    tool_call = part.function_call
                    result = self.call_function(tool_call)
                    history = f"{history}[我執行了 {tool_call.name}]\n"
                    function_response_part = types.Part.from_function_response(
                        name=tool_call.name,
                        response={"result": result},
                    )
                    result_parts.append(function_response_part)
                    
                else:
                    history = history + part.text

            if is_tools_call == False:
                return history
            else:
                self.contents.append(types.Content(role="user", parts=result_parts))                   
    

In [15]:
bot = weather_bot(API_KEY, model_name, config)

In [16]:
print(bot.chat("你好"))

您好！有什麼我可以幫助您的嗎？



In [17]:
print(bot.chat("我想要知道雪梨今天適合出門玩嗎"))

我執行了 get_coordinates
我執行了 get_weather
雪梨目前溫度為 21.1°C，很適合出門。



In [24]:
print(bot.chat("這樣的溫度會很冷嗎"))

對我來說，12.7度是涼爽的。要不要我幫你查詢一下，在這樣的溫度下，人們通常會穿什麼樣的衣服？



### 使用Gradio做介面

In [18]:
import gradio as gr
from google import genai

bot = weather_bot(API_KEY, model_name, config)
def chat_function(message, history):
    response = bot.chat(message)
    return response

demo = gr.ChatInterface(chat_function, type="messages", autofocus=False)

if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
